# SurvFace 03. Compressed materialization and pgvector indexes

02에서 SurvFace training-development로 학습한 frozen compressor를 같은 run의 전체 training 및 공식 test origin embedding에 적용합니다. PCA profile은 각 차원의 pgvector 테이블에 저장하고, PQ는 `pq_auxiliary` code로만 저장하며 pgvector HNSW vector로 취급하지 않습니다.

개발 통계 scan과 batch materialization은 약 10% 경계에서만 진행률을 출력합니다. 공식 test의 임베딩은 변환만 하며 compressor 또는 normalization 통계를 fit하지 않습니다.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(C:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from research.database import create_database_engine, load_database_settings
from research.experiments import materialize_survface_compressed_profiles
from research.runtime import ProgressReporter, RunStore, resolve_active_run

MODE = "real"
DATA_FRACTION = 1.0
SEED = 42
EXECUTE_STAGE = True
RUN_ROOT = PROJECT_ROOT / "runs" / "survface"
TRAINING_MANIFEST_PATH = PROJECT_ROOT / "data/interim/survface/training_manifest.csv"
RUN_DIR = resolve_active_run(
    RUN_ROOT,
    environment_variable="RONBUN_SURVFACE_RUN_DIR",
)
PROGRESS = ProgressReporter(
    "SurvFace 03 compressed materialization",
    heartbeat_seconds=None,
    milestone_percent=10,
)
preflight = {
    "mode": MODE,
    "data_fraction": DATA_FRACTION,
    "seed": SEED,
    "execute_stage": EXECUTE_STAGE,
    "run_dir": str(RUN_DIR),
    "training_manifest": str(TRAINING_MANIFEST_PATH),
}
preflight

{'mode': 'real',
 'data_fraction': 1.0,
 'seed': 42,
 'execute_stage': True,
 'run_dir': 'C:\\ronbun\\runs\\survface\\2026\\07\\29\\20260729-R001-3c19d8dc_thesis3_survface_official_v1',
 'training_manifest': 'C:\\ronbun\\data\\interim\\survface\\training_manifest.csv'}

In [2]:
result = {"status": "not_executed", **preflight}
if EXECUTE_STAGE:
    run = RunStore.open(RUN_DIR)
    config = run.config
    compression = config["compression"]
    pca_dimensions = tuple(int(value) for value in compression["pca"]["dimensions"])
    pq_settings = tuple(
        (int(item["m"]), int(item["nbits"]))
        for item in compression["pq"]["settings"]
    )
    materialized_pq = compression["pq"]["materialized_setting"]
    training_manifest = pd.read_csv(TRAINING_MANIFEST_PATH)
    engine = create_database_engine(load_database_settings())
    summary = materialize_survface_compressed_profiles(
        run,
        engine,
        training_manifest=training_manifest,
        project_root=PROJECT_ROOT,
        pca_dimensions=pca_dimensions,
        pq_settings=pq_settings,
        pq_materialization_setting=(
            int(materialized_pq["m"]),
            int(materialized_pq["nbits"]),
        ),
        batch_size=int(compression.get("batch_size", 512)),
        progress=PROGRESS.callback(key_prefix=f"{run.run_id}:"),
    )
    result = {
        "status": "completed",
        "run_id": run.run_id,
        "source_vectors": summary["primary"]["counts"]["source_vectors"],
        "pgvector_searchable_profiles": summary["pgvector_searchable_profiles"],
        "pq_auxiliary": summary["pq_auxiliary"],
        "fit_source": summary["fit_source"],
        "official_test_fit": summary["official_test_fit"],
    }
result

[04:25:52] SurvFace 03 compressed materialization | development normalization scan | elapsed=2m 11s | progress=10% processed=46592 total=463341 rate=354.35/s eta=19m 36s development_vectors=37367
[04:26:14] SurvFace 03 compressed materialization | development normalization scan | elapsed=2m 34s | progress=20% processed=92672 total=463341 rate=603.25/s eta=10m 14s development_vectors=74355
[04:26:38] SurvFace 03 compressed materialization | development normalization scan | elapsed=2m 57s | progress=30% processed=139264 total=463341 rate=785.68/s eta=6m 52s development_vectors=111691
[04:27:01] SurvFace 03 compressed materialization | development normalization scan | elapsed=3m 21s | progress=40% processed=185344 total=463341 rate=922.19/s eta=5m 01s development_vectors=148672
[04:27:25] SurvFace 03 compressed materialization | development normalization scan | elapsed=3m 45s | progress=50% processed=231936 total=463341 rate=1032.65/s eta=3m 44s development_vectors=176205
[04:27:48] SurvF

{'status': 'completed',
 'run_id': '20260729-R001-3c19d8dc',
 'source_vectors': 463341,
 'pgvector_searchable_profiles': ['origin_512',
  'pca_384',
  'pca_256',
  'pca_128',
  'pca_64',
  'pca_32'],
 'pq_auxiliary': {'profile': 'pq_512_m16_b8',
  'stored_as': 'pq_auxiliary',
  'pgvector_searchable': False},
 'fit_source': 'survface_training_development',
 'official_test_fit': False}

## 다음 단계

각 PCA row count가 origin source count와 같은지 확인합니다. `pq_auxiliary.pgvector_searchable=false`가 유지되어야 합니다. 이어서 `02_step1_compression_characterization.ipynb`와 `03_open_set/`을 순서대로 실행합니다.